# E3: missing-phase tests and a phase-dropout CNN

**Revision plan (approved 17 Oct 2026), experiment E3.** Answers Reviewer 4 points 3 and 4 and gives Reviewer 2
(point 6) a tested mitigation.

**Part A — controlled missing-phase test (no training).** The saved four-phase CNN (`checkpoints_w4`) is applied to the
same internal patients with phases removed one pattern at a time. A removed phase becomes an all-zero channel, exactly
what the external pipeline fed the model. Patterns (fixed in advance):

| Pattern | Phases kept | Why |
|---|---|---|
| complete | P, C1, C2, C3 | reference |
| drop_P / drop_C1 / drop_C2 / drop_C3 | three of four | one phase missing at a time |
| P+C2 | P, C2 | common external pattern (≈40 of 103 HCC-TACE-Seg patients) |
| C2_only | C2 | worst external pattern (5 of 103) |

`drop_C3` (no delayed phase) is the most common external pattern (57 of 103).

**Part B — retraining under a corrected protocol, two arms.** Both arms use the Week-4 architecture and settings
(MONAI 3D ResNet-18, 40 epochs, batch 4, Adam 1e-4, weight decay 1e-5, cosine schedule, mixed precision, same
augmentation). Two fixes apply to both arms: class weights from each fold's own training patients, and the checkpoint
chosen on an **inner validation split** (20% of the fold's training patients, stratified), so the outer validation fold
is used only for out-of-fold predictions.
- **R (retrained baseline):** no phase dropout.
- **D (phase dropout):** during training only, each sample keeps all phases with probability 0.5; otherwise each phase
  is dropped independently with probability 0.5 (at least one phase always kept). Dropped phases become zero channels.
  No extra "phase present" input is added, so the architecture is unchanged.
Checkpoint choice uses complete inputs for both arms.

**Part C — fusion.** CNN+radiomics and full fusion are refitted with each CNN (original, R, D) and the E2 no-shape
radiomics, then evaluated with complete inputs and with the CNN fed each missing-phase pattern (radiomics kept complete,
so the CNN's contribution is isolated).

**Reported for every model and pattern:** cross-validated macro-AUC (out-of-fold), held-out macro-AUC with 95% CI,
accuracy, macro-F1 and per-class recall; paired bootstrap differences. Nothing is tuned on the test set; every result is
reported.

**Runtime:** GPU required (Runtime → Change runtime type → A100 or L4). About 1–1.5 hours, mostly Part B training.
**If Colab disconnects:** set `RESUME_RUN_ID` in Cell 1 to the folder name printed earlier (e.g. `E3_20261018T090000Z`)
and run all cells again; finished folds are reloaded instead of retrained.

**Safety:** inputs are only read and checked. All outputs go to a new folder
`liver_project/revision_2026-10/E3_<time>/`; nothing existing is overwritten.

In [ ]:
# Cell 1 — Setup: paths, resume option, input checks, GPU, versions
RESUME_RUN_ID = None   # e.g. 'E3_20261018T090000Z' to continue an interrupted run

import os, sys, json, hashlib, platform, datetime, random, time, warnings, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(os.environ['COLAB_DRIVE_MOUNT'])  # see docs/revision_notebooks.md
    ROOT = Path(os.environ['LIVER_PROJECT_ROOT'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'monai'], check=True)
else:
    ROOT = Path(os.environ['E3_TEST_ROOT'])   # local test harness only

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import monai
from monai.networks.nets import resnet18

PROJECT_DIR = ROOT / 'liver_project'
DATA_DIR = ROOT / 'liver_data'
REVISION_DIR = PROJECT_DIR / 'revision_2026-10'
E2_DIR = REVISION_DIR / 'E1_E2_20261017T032941Z'
W4_DIR = PROJECT_DIR / 'checkpoints_w4'

SEED = 42
N_BOOT = 2000
CLASSES = ['HCC', 'ICC', 'CHCC']
PHASES = ['P', 'C1', 'C2', 'C3']
NUM_EPOCHS = int(os.environ.get('E3_TEST_EPOCHS', 40))   # 40 in Colab (env var exists only in the local test)

SMALL_INPUTS = {
    'patient_data': (DATA_DIR / 'patient_data.csv', '8cd4eb9405a3987c35d7ba4e92893c474f20b6a2791ab23bac2c47ef5532b185'),
    'split': (PROJECT_DIR / 'splits' / 'train_val_test.json', 'cfbe3cb6f9b1901c3d672042e7918a95051d841d328d8353492f91376617513e'),
    'fold_0': (PROJECT_DIR / 'splits' / 'fold_0.json', 'c5f32ebf1c646835055887e258dbc574aedbfea66bb197b4e6999961e255b02b'),
    'fold_1': (PROJECT_DIR / 'splits' / 'fold_1.json', '3726c0f6b915942b11519b4ab1b589fac3590ab4a0a1800b2fa72be8ad3a8391'),
    'fold_2': (PROJECT_DIR / 'splits' / 'fold_2.json', 'f50c29e35f728b7368885f0adf974410007203c0043ea581f2d75f1a2faefe33'),
    'fold_3': (PROJECT_DIR / 'splits' / 'fold_3.json', 'b740859b7bf5e3c3c8ce43707b5805b56ebb35453f732f1cd9a35369f584d37f'),
    'fold_4': (PROJECT_DIR / 'splits' / 'fold_4.json', '32deb2f725b24487ac447a4459d6e072ad90529b7ad7d537b0c73a3f342cd7b8'),
    'w4_oof': (PROJECT_DIR / 'fusion' / 'week4_cnn_oof.csv', '5e74804369fdec9235294272956c139174fb99bc8a0ed518032059e090d20504'),
    'w4_test': (PROJECT_DIR / 'results' / 'week4_test_predictions.csv', '3ea371dd804def54bd9d56e91a713f0d3fe9e2d9bad12cc7d52331acc385c2b0'),
    'e2_oof_meta': (E2_DIR / 'E2_fusion_oof_meta_features_noshape.csv', 'c20d23d3d0a7df871212f6f937bc2024e686c11dc6975350329cbe186f058d12'),
    'e2_test_meta': (E2_DIR / 'E2_fusion_test_meta_features_noshape.csv', 'fbb946f641a050f66b093564141a8e103e1c4e5c651d6abeb51042293ab8cb2f'),
}
LARGE_INPUTS = {   # checked by size here, and by reproducing their saved predictions in Cell 4
    'processed_tar': (PROJECT_DIR / 'processed.tar', 402616320),
    **{f'w4_fold_{f}': (W4_DIR / f'fold_{f}_best.pt', 132990161) for f in range(5)},
}

def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

problems, input_hashes = [], {}
for key, (path, expected) in SMALL_INPUTS.items():
    if not path.exists():
        problems.append(f'missing: {path}')
        continue
    input_hashes[key] = sha256_of(path)
    if input_hashes[key] != expected:
        problems.append(f'changed: {path.name}')
for key, (path, size) in LARGE_INPUTS.items():
    if not path.exists():
        problems.append(f'missing: {path}')
    elif IN_COLAB and path.stat().st_size != size:
        problems.append(f'unexpected size: {path.name}')
if problems:
    raise RuntimeError('STOP — input check failed:\n' + '\n'.join(problems))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if IN_COLAB and DEVICE.type != 'cuda':
    raise RuntimeError('Select a GPU runtime: Runtime > Change runtime type > A100 or L4 GPU.')
USE_AMP = DEVICE.type == 'cuda'
torch.backends.cudnn.benchmark = True

import sklearn, scipy
VERSIONS = {'python': platform.python_version(), 'torch': torch.__version__, 'monai': monai.__version__,
            'numpy': np.__version__, 'pandas': pd.__version__, 'scikit-learn': sklearn.__version__,
            'scipy': scipy.__version__, 'device': torch.cuda.get_device_name(0) if USE_AMP else 'cpu'}

if RESUME_RUN_ID:
    OUT_DIR = REVISION_DIR / RESUME_RUN_ID
    if not OUT_DIR.is_dir():
        raise FileNotFoundError(f'Cannot resume: {OUT_DIR} does not exist.')
    RUN_ID = RESUME_RUN_ID.replace('E3_', '')
else:
    RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    OUT_DIR = REVISION_DIR / f'E3_{RUN_ID}'
    OUT_DIR.mkdir(parents=False, exist_ok=False)

print('Inputs checked:', len(SMALL_INPUTS) + len(LARGE_INPUTS))
print('Versions:', VERSIONS)
print('Output folder:', OUT_DIR.name, '(resumed)' if RESUME_RUN_ID else '(new)')

In [ ]:
# Cell 2 — Stage the preprocessed crops locally and load them into memory (read-only)
if IN_COLAB:
    PROCESSED_DIR = Path('/content/processed')
    if len(list(PROCESSED_DIR.glob('*.npz'))) < 278:
        subprocess.run(['cp', str(LARGE_INPUTS['processed_tar'][0]), '/content/processed.tar'], check=True)
        subprocess.run(['tar', '-xf', '/content/processed.tar', '-C', '/content/'], check=True)
        os.remove('/content/processed.tar')
else:
    PROCESSED_DIR = Path(os.environ['E3_TEST_PROCESSED'])

split = json.loads(SMALL_INPUTS['split'][0].read_text())
dev_ids = [str(x) for x in split['dev_ids']]
test_ids = [str(x) for x in split['test_ids']]
fold_splits = [json.loads(SMALL_INPUTS[f'fold_{f}'][0].read_text()) for f in range(5)]
assert len(dev_ids) == 222 and len(test_ids) == 56

CT, LABEL = {}, {}
for pid in dev_ids + test_ids:
    with np.load(PROCESSED_DIR / f'{pid}.npz', allow_pickle=False) as z:
        CT[pid] = z['ct'].astype(np.uint8)
        LABEL[pid] = int(z['label'])
shapes = {v.shape for v in CT.values()}
assert len(CT) == 278 and len(shapes) == 1 and next(iter(shapes))[0] == 4, shapes
print('Loaded crops for 278 patients, shape', next(iter(shapes)))
print('Class counts dev / test:', np.bincount([LABEL[p] for p in dev_ids]).tolist(),
      np.bincount([LABEL[p] for p in test_ids]).tolist())

In [ ]:
# Cell 3 — Shared definitions (normalization, augmentation and model copied from Week 4)
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, recall_score

PATTERNS = {
    'complete': (0, 1, 2, 3),
    'drop_P': (1, 2, 3),
    'drop_C1': (0, 2, 3),
    'drop_C2': (0, 1, 3),
    'drop_C3': (0, 1, 2),
    'P+C2': (0, 2),
    'C2_only': (2,),
}

def zscore_nonzero_per_phase(x4):          # Week 4, unchanged
    out = np.zeros_like(x4, dtype=np.float32)
    for c in range(x4.shape[0]):
        vol = x4[c]
        mask = vol > 0
        if mask.sum() < 100:
            out[c] = vol.astype(np.float32)
            continue
        vals = vol[mask]
        mean, std = vals.mean(), vals.std() + 1e-6
        out[c][mask] = (vals - mean) / std
    return out

def augment(vol4):                           # Week 4, unchanged
    if random.random() < 0.5:
        vol4 = vol4[:, ::-1].copy()
    if random.random() < 0.5:
        vol4 = vol4[:, :, ::-1].copy()
    if random.random() < 0.5:
        for c in range(vol4.shape[0]):
            scale = 1.0 + (random.random() - 0.5) * 0.2
            shift = (random.random() - 0.5) * 0.2
            mask = vol4[c] != 0
            vol4[c][mask] = vol4[c][mask] * scale + shift
    return vol4

def sample_kept_phases():
    """Phase dropout (prespecified): complete with p=0.5, else drop each phase with p=0.5, keep >=1."""
    if random.random() < 0.5:
        return (0, 1, 2, 3)
    while True:
        kept = tuple(c for c in range(4) if random.random() >= 0.5)
        if kept:
            return kept

def keep_only(vol4, kept):
    out = vol4.copy()
    for c in range(4):
        if c not in kept:
            out[c] = 0.0
    return out

class CropDataset(Dataset):
    def __init__(self, patient_ids, train=False, phase_dropout=False, kept=(0, 1, 2, 3)):
        self.ids, self.train, self.phase_dropout, self.kept = list(patient_ids), train, phase_dropout, kept
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, index):
        pid = self.ids[index]
        vol4 = zscore_nonzero_per_phase(CT[pid].astype(np.float32))
        if self.train:
            vol4 = augment(vol4)
        kept = sample_kept_phases() if (self.train and self.phase_dropout) else self.kept
        if kept != (0, 1, 2, 3):
            vol4 = keep_only(vol4, kept)
        return torch.from_numpy(np.ascontiguousarray(vol4, dtype=np.float32)), LABEL[pid], pid

def make_model():                            # Week 4, unchanged
    return resnet18(spatial_dims=3, n_input_channels=4, num_classes=3, feed_forward=True)

def class_weights_for(ids):                  # Week 4 formula, applied to this fold's training patients only
    counts = np.bincount([LABEL[p] for p in ids], minlength=3).astype(np.float32)
    return torch.from_numpy(counts.sum() / (3 * counts.clip(min=1))).float()

@torch.no_grad()
def predict(model, ids, kept=(0, 1, 2, 3), batch_size=8):
    model.eval()
    loader = DataLoader(CropDataset(ids, kept=kept), batch_size=batch_size, shuffle=False,
                        num_workers=2 if IN_COLAB else 0, pin_memory=USE_AMP)
    probs, pids = [], []
    for x, _, batch_ids in loader:
        probs.append(torch.softmax(model(x.to(DEVICE, non_blocking=True)).float().cpu(), dim=1).numpy())
        pids.extend(batch_ids)
    return pd.DataFrame(np.concatenate(probs), index=pids, columns=['p_HCC', 'p_ICC', 'p_CHCC'])

def macro_auc(y, p):
    return float(roc_auc_score(y, p, multi_class='ovr', average='macro'))

def bootstrap_ci(y, p, seed=SEED):
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 3:
            values.append(macro_auc(y[i], p[i]))
    return float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5))

def paired_delta(y, p_a, p_b, seed=SEED):
    rng = np.random.default_rng(seed)
    deltas = []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 3:
            deltas.append(macro_auc(y[i], p_a[i]) - macro_auc(y[i], p_b[i]))
    deltas = np.asarray(deltas)
    return {'delta': macro_auc(y, p_a) - macro_auc(y, p_b), 'ci_low': float(np.percentile(deltas, 2.5)),
            'ci_high': float(np.percentile(deltas, 97.5)),
            'bootstrap_two_sided_p': float(min(1.0, 2 * min((deltas <= 0).mean(), (deltas >= 0).mean())))}

FOLD_OF = {pid: f for f, s in enumerate(fold_splits) for pid in s['val_ids']}
Y_DEV = np.array([LABEL[p] for p in dev_ids])
Y_TEST = np.array([LABEL[p] for p in test_ids])

def evaluate_predictions(oof, test, model_name, pattern):
    """oof: DataFrame indexed by dev patient; test: DataFrame indexed by test patient."""
    p_oof = oof.loc[dev_ids].to_numpy()
    folds = np.array([FOLD_OF[p] for p in dev_ids])
    fold_auc = [macro_auc(Y_DEV[folds == f], p_oof[folds == f]) for f in range(5)]
    p_test = test.loc[test_ids].to_numpy()
    pred = p_test.argmax(axis=1)
    low, high = bootstrap_ci(Y_TEST, p_test)
    recalls = recall_score(Y_TEST, pred, labels=[0, 1, 2], average=None, zero_division=0)
    return {'model': model_name, 'pattern': pattern, 'cv_macro_auc_mean': float(np.mean(fold_auc)),
            'cv_macro_auc_sd': float(np.std(fold_auc, ddof=1)), 'test_macro_auc': macro_auc(Y_TEST, p_test),
            'test_ci_low': low, 'test_ci_high': high, 'test_accuracy': float(accuracy_score(Y_TEST, pred)),
            'test_macro_f1': float(f1_score(Y_TEST, pred, average='macro', zero_division=0)),
            'recall_HCC': float(recalls[0]), 'recall_ICC': float(recalls[1]), 'recall_CHCC': float(recalls[2])}

def predict_all_patterns(fold_models, model_name):
    """Out-of-fold (each dev patient by the fold model that never saw it) and 5-model test ensemble, per pattern."""
    oof_by_pattern, test_by_pattern = {}, {}
    for pattern, kept in PATTERNS.items():
        oof = pd.concat([predict(fold_models[f], fold_splits[f]['val_ids'], kept) for f in range(5)])
        test = sum(predict(fold_models[f], test_ids, kept) for f in range(5)) / 5.0
        oof_by_pattern[pattern], test_by_pattern[pattern] = oof, test
    return oof_by_pattern, test_by_pattern

def long_predictions(oof_by_pattern, test_by_pattern, model_name):
    rows = []
    for pattern in PATTERNS:
        for split_name, table in [('oof', oof_by_pattern[pattern]), ('test', test_by_pattern[pattern])]:
            t = table.copy()
            t.insert(0, 'patient_id', t.index)
            t.insert(1, 'split', split_name)
            t.insert(0, 'pattern', pattern)
            t.insert(0, 'model', model_name)
            rows.append(t)
    return pd.concat(rows, ignore_index=True)

def load_fold_model(path):
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    model = make_model().to(DEVICE)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    return model

print('Definitions ready. Patterns:', list(PATTERNS))

In [ ]:
# Cell 4 — Part A integrity check: the saved Week-4 models must reproduce their saved predictions
original_models = [load_fold_model(LARGE_INPUTS[f'w4_fold_{f}'][0]) for f in range(5)]
check_oof = pd.concat([predict(original_models[f], fold_splits[f]['val_ids']) for f in range(5)])
check_test = sum(predict(original_models[f], test_ids) for f in range(5)) / 5.0

saved_oof = pd.read_csv(SMALL_INPUTS['w4_oof'][0], dtype={'patient_id': str}).set_index('patient_id')
saved_test = pd.read_csv(SMALL_INPUTS['w4_test'][0], dtype={'patient_id': str}).set_index('patient_id')
oof_diff = float(np.abs(check_oof.loc[saved_oof.index].to_numpy()
                        - saved_oof[['w4_p_HCC', 'w4_p_ICC', 'w4_p_CHCC']].to_numpy()).max())
test_diff = float(np.abs(check_test.loc[saved_test.index].to_numpy()
                         - saved_test[['prob_HCC', 'prob_ICC', 'prob_CHCC']].to_numpy()).max())
auc_test_saved = macro_auc(Y_TEST, saved_test.loc[test_ids, ['prob_HCC', 'prob_ICC', 'prob_CHCC']].to_numpy())
auc_test_now = macro_auc(Y_TEST, check_test.loc[test_ids].to_numpy())
auc_oof_saved = macro_auc(Y_DEV, saved_oof.loc[dev_ids, ['w4_p_HCC', 'w4_p_ICC', 'w4_p_CHCC']].to_numpy())
auc_oof_now = macro_auc(Y_DEV, check_oof.loc[dev_ids].to_numpy())
auc_diff = max(abs(auc_test_now - auc_test_saved), abs(auc_oof_now - auc_oof_saved))
# GPU type and TF32 maths can shift probabilities slightly; the AUC must stay essentially identical.
A_INTEGRITY_PASS = max(oof_diff, test_diff) < 0.02 and auc_diff < 0.005
print(f'Largest probability difference from saved Week-4 predictions: out-of-fold {oof_diff:.2e}, held-out {test_diff:.2e}')
print(f'Macro-AUC saved vs recomputed: held-out {auc_test_saved:.4f} vs {auc_test_now:.4f}; '
      f'pooled out-of-fold {auc_oof_saved:.4f} vs {auc_oof_now:.4f}')
if not A_INTEGRITY_PASS:
    message = 'The saved CNN models do not reproduce their saved predictions.'
    if IN_COLAB:
        raise RuntimeError('STOP — ' + message + ' Send this output for review.')
    print('LOCAL TEST:', message, '(expected with the stand-in model)')
else:
    print('Part A integrity PASS: saved models and crops reproduce the submitted CNN predictions.')

In [ ]:
# Cell 5 — Part A: the ORIGINAL CNN under each missing-phase pattern (no training)
t0 = time.time()
oof_orig, test_orig = predict_all_patterns(original_models, 'original')
results_A = pd.DataFrame([evaluate_predictions(oof_orig[p], test_orig[p], 'original', p) for p in PATTERNS])
deltas_A = pd.DataFrame([{'model': 'original', 'comparison': f'{p} - complete',
                          **paired_delta(Y_TEST, test_orig[p].loc[test_ids].to_numpy(),
                                         test_orig['complete'].loc[test_ids].to_numpy())}
                         for p in PATTERNS if p != 'complete'])
results_A.to_csv(OUT_DIR / 'E3A_original_cnn_by_pattern.csv', index=False)
deltas_A.to_csv(OUT_DIR / 'E3A_original_cnn_deltas_vs_complete.csv', index=False)
long_predictions(oof_orig, test_orig, 'original').to_csv(OUT_DIR / 'E3A_original_cnn_predictions.csv', index=False)
del original_models
if USE_AMP:
    torch.cuda.empty_cache()
print(f'Part A done in {(time.time() - t0) / 60:.1f} min')
print(results_A[['pattern', 'cv_macro_auc_mean', 'test_macro_auc', 'test_ci_low', 'test_ci_high',
                 'test_accuracy', 'recall_HCC', 'recall_ICC', 'recall_CHCC']].round(3).to_string(index=False))

In [ ]:
# Cell 6 — Part B: train arm R (no dropout) and arm D (phase dropout) under the corrected protocol
BATCH_SIZE, LR, WD = 4, 1e-4, 1e-5          # Week 4 settings
ARMS = {'R_retrained': False, 'D_phase_dropout': True}

def set_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if USE_AMP:
        torch.cuda.manual_seed_all(seed)

def train_fold(arm, phase_dropout, fold):
    path = OUT_DIR / arm / f'fold_{fold}.pt'
    if path.exists():
        print(f'  {arm} fold {fold}: already trained, loading.')
        return
    path.parent.mkdir(exist_ok=True)
    outer_train = list(fold_splits[fold]['train_ids'])
    inner_train, inner_val = train_test_split(outer_train, test_size=0.2, random_state=SEED + fold,
                                              stratify=[LABEL[p] for p in outer_train])
    set_seeds(SEED + fold)
    workers = 2 if IN_COLAB else 0
    train_loader = DataLoader(CropDataset(inner_train, train=True, phase_dropout=phase_dropout),
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=workers,
                              pin_memory=USE_AMP, drop_last=True)
    model = make_model().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    criterion = nn.CrossEntropyLoss(weight=class_weights_for(inner_train).to(DEVICE))
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    y_inner = np.array([LABEL[p] for p in inner_val])
    best_auc, best_state, best_epoch, history = -1.0, None, -1, []
    for epoch in range(NUM_EPOCHS):
        model.train()
        started, loss_sum, n = time.time(), 0.0, 0
        for x, y, _ in train_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * x.size(0)
            n += x.size(0)
        scheduler.step()
        inner_auc = macro_auc(y_inner, predict(model, inner_val).loc[inner_val].to_numpy())
        history.append({'epoch': epoch, 'train_loss': loss_sum / max(n, 1), 'inner_val_auc': inner_auc,
                        'seconds': time.time() - started})
        if inner_auc > best_auc:
            best_auc, best_epoch = inner_auc, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    torch.save({'state_dict': best_state, 'arm': arm, 'fold': fold, 'best_epoch': best_epoch,
                'best_inner_val_auc': best_auc, 'inner_train_ids': inner_train, 'inner_val_ids': inner_val,
                'phase_dropout': phase_dropout, 'epochs': NUM_EPOCHS}, path)
    pd.DataFrame(history).to_csv(OUT_DIR / arm / f'fold_{fold}_history.csv', index=False)
    print(f'  {arm} fold {fold}: best inner-val AUC {best_auc:.4f} at epoch {best_epoch} '
          f'({sum(h["seconds"] for h in history) / 60:.1f} min)')
    del model
    if USE_AMP:
        torch.cuda.empty_cache()

t0 = time.time()
for arm, phase_dropout in ARMS.items():
    print(arm)
    for fold in range(5):
        train_fold(arm, phase_dropout, fold)
print(f'Part B training done in {(time.time() - t0) / 60:.1f} min')

In [ ]:
# Cell 7 — Part B: evaluate both arms under every pattern and compare
oof_arm, test_arm, results_B = {}, {}, []
for arm in ARMS:
    models = [load_fold_model(OUT_DIR / arm / f'fold_{f}.pt') for f in range(5)]
    oof_arm[arm], test_arm[arm] = predict_all_patterns(models, arm)
    results_B += [evaluate_predictions(oof_arm[arm][p], test_arm[arm][p], arm, p) for p in PATTERNS]
    long_predictions(oof_arm[arm], test_arm[arm], arm).to_csv(OUT_DIR / f'E3B_{arm}_predictions.csv', index=False)
    del models
    if USE_AMP:
        torch.cuda.empty_cache()
results_B = pd.DataFrame(results_B)

def test_matrix(source, pattern):
    return source[pattern].loc[test_ids].to_numpy()

delta_rows = []
for p in PATTERNS:
    delta_rows.append({'comparison': f'D - R ({p})', **paired_delta(
        Y_TEST, test_matrix(test_arm['D_phase_dropout'], p), test_matrix(test_arm['R_retrained'], p))})
    delta_rows.append({'comparison': f'D - original ({p})', **paired_delta(
        Y_TEST, test_matrix(test_arm['D_phase_dropout'], p), test_matrix(test_orig, p))})
delta_rows.append({'comparison': 'R - original (complete)', **paired_delta(
    Y_TEST, test_matrix(test_arm['R_retrained'], 'complete'), test_matrix(test_orig, 'complete'))})
deltas_B = pd.DataFrame(delta_rows)

results_all = pd.concat([results_A, results_B], ignore_index=True)
results_all.to_csv(OUT_DIR / 'E3_all_cnns_by_pattern.csv', index=False)
deltas_B.to_csv(OUT_DIR / 'E3B_paired_deltas.csv', index=False)
pivot = results_all.pivot(index='pattern', columns='model', values='test_macro_auc').loc[list(PATTERNS)]
print('Held-out macro-AUC by pattern:')
print(pivot[['original', 'R_retrained', 'D_phase_dropout']].round(3).to_string())
print('\nCross-validated macro-AUC by pattern:')
print(results_all.pivot(index='pattern', columns='model', values='cv_macro_auc_mean')
      .loc[list(PATTERNS), ['original', 'R_retrained', 'D_phase_dropout']].round(3).to_string())
print('\nHCC recall by pattern (relevant to the external HCC cohort):')
print(results_all.pivot(index='pattern', columns='model', values='recall_HCC')
      .loc[list(PATTERNS), ['original', 'R_retrained', 'D_phase_dropout']].round(3).to_string())

In [ ]:
# Cell 8 — Part C: fusion with each CNN (E2 no-shape radiomics), complete and missing-phase CNN inputs
oof_meta = pd.read_csv(SMALL_INPUTS['e2_oof_meta'][0], dtype={'patient_id': str})
test_meta = pd.read_csv(SMALL_INPUTS['e2_test_meta'][0], dtype={'patient_id': str})
W4 = ['w4_p_HCC', 'w4_p_ICC', 'w4_p_CHCC']
W5 = ['w5_p_HCC', 'w5_p_ICC', 'w5_p_CHCC']
FEATURES = {'cnn_plus_radiomics': W4 + W5, 'full_fusion': W4 + W5 + ['age', 'gender']}

def with_cnn(frame, cnn_predictions):
    out = frame.copy()
    out[W4] = cnn_predictions.loc[out['patient_id']].to_numpy()
    return out

def prepare(matrix):
    matrix = matrix.astype(np.float32)
    matrix[matrix <= -1] = np.nan
    return matrix

def fit_meta_folds(columns, oof_frame):        # same model as the submitted fusion (fit on complete inputs)
    fitted = []
    for f in range(5):
        train = oof_frame['fold'].astype(int).ne(f)
        X = prepare(oof_frame.loc[train, columns].to_numpy())
        imputer = SimpleImputer(strategy='median').fit(X)
        scaler = StandardScaler().fit(imputer.transform(X))
        model = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED)
        model.fit(scaler.transform(imputer.transform(X)), oof_frame.loc[train, 'label'].astype(int))
        fitted.append((imputer, scaler, model))
    return fitted

def apply_meta_folds(fitted, columns, oof_frame, test_frame):
    oof_probs = pd.DataFrame(index=oof_frame['patient_id'], columns=['p_HCC', 'p_ICC', 'p_CHCC'], dtype=float)
    test_probs = 0.0
    for f, (imputer, scaler, model) in enumerate(fitted):
        val = oof_frame['fold'].astype(int).eq(f)
        X_val = scaler.transform(imputer.transform(prepare(oof_frame.loc[val, columns].to_numpy())))
        oof_probs.loc[oof_frame.loc[val, 'patient_id']] = model.predict_proba(X_val)
        test_probs = test_probs + model.predict_proba(
            scaler.transform(imputer.transform(prepare(test_frame[columns].to_numpy())))) / 5.0
    return oof_probs, pd.DataFrame(test_probs, index=test_frame['patient_id'].values,
                                   columns=['p_HCC', 'p_ICC', 'p_CHCC'])

cnn_sources = {'original': (oof_orig, test_orig), 'R_retrained': (oof_arm['R_retrained'], test_arm['R_retrained']),
               'D_phase_dropout': (oof_arm['D_phase_dropout'], test_arm['D_phase_dropout'])}
results_C, fusion_test = [], {}
for cnn_name, (oof_cnn, test_cnn) in cnn_sources.items():
    base_oof, base_test = with_cnn(oof_meta, oof_cnn['complete']), with_cnn(test_meta, test_cnn['complete'])
    for fusion_name, columns in FEATURES.items():
        fitted = fit_meta_folds(columns, base_oof)
        for p in PATTERNS:
            oof_p, test_p = apply_meta_folds(fitted, columns, with_cnn(oof_meta, oof_cnn[p]), with_cnn(test_meta, test_cnn[p]))
            fusion_test[(cnn_name, fusion_name, p)] = test_p
            results_C.append(evaluate_predictions(oof_p, test_p, f'{fusion_name} [{cnn_name} CNN]', p))
results_C = pd.DataFrame(results_C)

# consistency: original CNN + complete inputs must reproduce the E2 fusion results
e2_main = pd.read_csv(E2_DIR / 'E2_main_results.csv').query("analysis == 'E2_no_shape_main'").set_index('model_key')
for fusion_name in FEATURES:
    row = results_C[(results_C['model'] == f'{fusion_name} [original CNN]') & (results_C['pattern'] == 'complete')].iloc[0]
    expected = e2_main.loc[fusion_name, 'test_macro_auc']
    print(f'Consistency {fusion_name}: E3 original-CNN complete {row["test_macro_auc"]:.4f} vs E2 {expected:.4f}')
    if A_INTEGRITY_PASS and abs(row['test_macro_auc'] - expected) > 5e-3:
        raise RuntimeError('STOP — fusion with the original CNN does not match E2. Send this output for review.')

fusion_deltas = []
for fusion_name in FEATURES:
    for p in PATTERNS:
        fusion_deltas.append({'comparison': f'{fusion_name}: D-CNN - original-CNN ({p})', **paired_delta(
            Y_TEST, fusion_test[('D_phase_dropout', fusion_name, p)].loc[test_ids].to_numpy(),
            fusion_test[('original', fusion_name, p)].loc[test_ids].to_numpy())})
fusion_deltas = pd.DataFrame(fusion_deltas)
results_C.to_csv(OUT_DIR / 'E3C_fusion_by_cnn_and_pattern.csv', index=False)
fusion_deltas.to_csv(OUT_DIR / 'E3C_fusion_paired_deltas.csv', index=False)
print('Held-out macro-AUC of FULL FUSION by CNN and pattern (radiomics complete):')
print(results_C[results_C['model'].str.startswith('full_fusion')]
      .pivot(index='pattern', columns='model', values='test_macro_auc').loc[list(PATTERNS)].round(3).to_string())

In [ ]:
# Cell 9 — Summary, manifest and the block to send back
summary = {
    'run_id': RUN_ID, 'notebook': 'E3_missing_phases_and_phase_dropout.ipynb', 'versions': VERSIONS,
    'input_sha256': input_hashes, 'part_A_integrity_pass': A_INTEGRITY_PASS,
    'part_A_max_diff': {'oof': oof_diff, 'test': test_diff, 'auc': auc_diff},
    'patterns': {k: [PHASES[c] for c in v] for k, v in PATTERNS.items()},
    'protocol': {'epochs': NUM_EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LR, 'weight_decay': WD,
                 'inner_validation': '20% of fold training patients, stratified, seed 42 + fold',
                 'class_weights': 'from inner-training patients of each fold',
                 'phase_dropout': 'p(complete)=0.5, else each phase dropped with p=0.5, >=1 kept; zero channel',
                 'checkpoint_choice': 'best inner-validation macro-AUC on complete inputs',
                 'test_set_used_for_tuning': False},
    'best_epochs': {arm: [int(torch.load(OUT_DIR / arm / f'fold_{f}.pt', map_location='cpu',
                                         weights_only=False)['best_epoch']) for f in range(5)] for arm in ARMS},
}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))
manifest = [{'file': p.relative_to(OUT_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of(p)}
            for p in sorted(OUT_DIR.rglob('*')) if p.is_file() and p.name != 'output_manifest.csv']
pd.DataFrame(manifest).to_csv(OUT_DIR / 'output_manifest.csv', index=False)

print('\n================ COPY FROM HERE AND SEND BACK ================')
print(f'Run E3_{RUN_ID} | Part A integrity pass: {A_INTEGRITY_PASS} '
      f'(max diff OOF {oof_diff:.1e}, test {test_diff:.1e}) | best epochs: {summary["best_epochs"]}')
print('\nCNN held-out macro-AUC by pattern:')
print(pivot[['original', 'R_retrained', 'D_phase_dropout']].round(3).to_string())
print('\nCNN HCC recall by pattern:')
print(results_all.pivot(index='pattern', columns='model', values='recall_HCC')
      .loc[list(PATTERNS), ['original', 'R_retrained', 'D_phase_dropout']].round(3).to_string())
print('\nKey paired differences (held-out macro-AUC):')
print(deltas_B[deltas_B['comparison'].str.contains('complete|drop_C3|P\\+C2|C2_only')]
      [['comparison', 'delta', 'ci_low', 'ci_high']].round(3).to_string(index=False))
print('\nFull fusion held-out macro-AUC by CNN and pattern:')
print(results_C[results_C['model'].str.startswith('full_fusion')]
      .pivot(index='pattern', columns='model', values='test_macro_auc').loc[list(PATTERNS)].round(3).to_string())
print('\nVersions:', VERSIONS)
print('==============================================================')